# DeepLOB → FPGA: training on Colab GPU

1. **Runtime → Change runtime type → T4 GPU**, then Save.
2. Run the cells from top to bottom (Shift+Enter).
3. Everything is saved in Google Drive (`MyDrive/deeplob-fpga`), so if Colab disconnects, reconnect,
   run cells 1–3 again, and skip the runs whose `.json` already exists (cell 5 does that automatically).

## 1. GPU check + Google Drive

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

## 2. Code and data
The first time, a file picker opens: choose the **deeplob-fpga.zip** you downloaded (the newest one).
Later sessions skip the upload because the folder is already in Drive.

In [ ]:
import os, zipfile
ROOT = '/content/drive/MyDrive'
PROJ = f'{ROOT}/deeplob-fpga'
if not os.path.exists(f'{PROJ}/train.py'):
    from google.colab import files
    up = files.upload()                       # pick deeplob-fpga.zip
    name = next(iter(up))
    zipfile.ZipFile(name).extractall(ROOT)
    os.remove(name)
os.chdir(PROJ)
if not os.path.exists('data/Train_Dst_NoAuction_DecPre_CF_7.txt') and not os.path.exists('data/Train_Dst_NoAuction_DecPre_CF_7.npy'):
    !curl -L -o /content/data.zip https://raw.githubusercontent.com/zcakhaa/DeepLOB-Deep-Convolutional-Neural-Networks-for-Limit-Order-Books/master/data/data.zip
    !unzip -o -q /content/data.zip -d data
!ls data
print(os.getcwd())

## 3. Packages (PyTorch is already installed on Colab)

In [ ]:
!pip install -q brevitas onnx onnxscript
import torch; print(torch.__version__, 'cuda:', torch.cuda.is_available())

## 4. Smoke test (about 1 minute)

In [ ]:
!python train.py --model light --epochs 2 --train-stride 20 --test-stride 20 --tag smoke

## 5. Full experiments
Each run is skipped if its result file already exists, so this cell can simply be run again after a disconnect.
Logs go to `logs/`. Order: LightLOB FP32 → QAT 8/4/2 bit (from the FP32 weights) → DeepLOB and the no-LSTM ablation.
Edit `HORIZONS` and `SEEDS` if needed (the proposal needs 3 seeds; start with one).

In [ ]:
import subprocess, os, sys
HORIZONS = [10, 100]
SEEDS = [0]
EPOCHS = 50          # early stopping (patience 10) usually ends sooner
os.makedirs('logs', exist_ok=True)

def run(tag, args):
    if os.path.exists(f'runs/{tag}.json'):
        print(f'skip {tag} (done)'); return
    cmd = f'python train.py {args}'
    print('>>', cmd, flush=True)
    with open(f'logs/{tag}.log', 'w') as log:
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end='', flush=True); log.write(line); log.flush()
        if p.wait() != 0:
            raise RuntimeError(f'{tag} failed, see logs/{tag}.log')

for K in HORIZONS:
    for S in SEEDS:
        base = f'--horizon {K} --seed {S} --epochs {EPOCHS}'
        run(f'light_fp32_k{K}_s{S}', f'--model light {base}')
        for B in (8, 4, 2):
            run(f'light_w{B}a{B}_k{K}_s{S}',
                f'--model light --bits {B} {base} --init runs/light_fp32_k{K}_s{S}.pt --lr 5e-4')
        run(f'deeplob_fp32_k{K}_s{S}', f'--model deeplob {base}')
        run(f'deeplob_nolstm_fp32_k{K}_s{S}', f'--model deeplob_nolstm {base}')

## 6. Summary tables, shift analysis, QONNX export

In [ ]:
!python measure.py --runs runs
!python shift_eval.py --runs "runs/*.json"
!python export_qonnx.py --ckpt runs/light_w4a4_k10_s0.pt --bits 4 --out finn/light_w4a4.onnx
!ls results results/shift finn

## 7. Quick overview of all runs

In [ ]:
import json, glob, pandas as pd
rows = []
for f in sorted(glob.glob('runs/*.json')):
    r = json.load(open(f))
    if 'test' not in r: continue
    rows.append({'run': r['tag'], 'params': r['params'], 'best_ep': r['best_epoch'],
                 'val_f1': round(r['val']['f1_macro'], 4), 'test_f1': round(r['test']['f1_macro'], 4),
                 'test_acc': round(r['test']['accuracy'], 4), 'minutes': round(r['train_seconds'] / 60, 1)})
pd.DataFrame(rows)

## 8. Download results to the laptop
Everything is already in Drive (`MyDrive/deeplob-fpga`). This also makes one zip without the data.

In [ ]:
!cd .. && zip -qr /content/deeplob_results.zip deeplob-fpga/runs deeplob-fpga/results deeplob-fpga/logs deeplob-fpga/finn
from google.colab import files
files.download('/content/deeplob_results.zip')